# 第14章　前処理パイプラインの実装：入力画像から学習用のテンソルまで

**『医療診断支援AI開発　第2巻　応用編 ― 本格実装から臨床導入まで（第2巻　応用編）』のコード**

本文に載っているコードを、章の順にそのまま収めています。紙面のコードは読んで理解するためのもの、こちらは動かすためのものです。

- Python 以外（シェル・YAML・Dockerfile など）は、実行環境が違うので**コードセルにせず、そのまま読める形で置いています**。使う場所を確かめてから実行してください。
- 抜粋である以上、上から順に実行するだけで通るとは限りません。データの取得先やパスは、お手元の環境に合わせてください。
- **教育・研究のためのコードです。患者データをこのノートブックに置かないでください。**

リポジトリ: https://github.com/kewel-corp/book-vol2

## 14.1　2次元CT：DICOMからテンソルへ

In [ ]:
import pydicom, numpy as np, torch, cv2

def load_dicom_hu(path):
    """入門編のモダリティ別の前処理の章と同じ関数（章単位で動かせるよう再掲）。CT専用。
    HUと確認できない入力は処理を進めず、例外を発生させる。"""
    ds = pydicom.dcmread(path)
    if getattr(ds, "Modality", "") != "CT":
        raise ValueError("CT以外のモダリティ。CT値は定義されない（別の正規化の経路へ）")
    if int(getattr(ds, "NumberOfFrames", 1)) != 1:
        raise ValueError("マルチフレーム。別の変換経路を確認してから扱う")
    if "RescaleSlope" not in ds or "RescaleIntercept" not in ds:
        raise ValueError("CT値への変換係数がない。既定値で埋めず、変換不能として扱う")
    rescale_type = str(getattr(ds, "RescaleType", "")).strip().upper()
    image_type = list(getattr(ds, "ImageType", []))
    multi_energy = str(ds[(0x0018, 0x9361)].value).strip() if (0x0018, 0x9361) in ds else "NO"
    hu_by_image_type = (len(image_type) >= 3 and image_type[0] == "ORIGINAL"
                        and image_type[2] != "LOCALIZER" and multi_energy == "NO")
    if rescale_type != "HU" and not (not rescale_type and hu_by_image_type):
        raise ValueError("変換後の単位がHUと確認できない。別の変換経路で扱う")
    arr = ds.pixel_array.astype(np.float32)
    return arr * float(ds.RescaleSlope) + float(ds.RescaleIntercept), ds   # 格納値→CT値

def preprocess_ct_2d(dcm_path, window=(40, 400), size=512):
    # ① 保存された画素値をCT値に変換。CT・単一フレーム・変換係数・HU単位を確かめてから（確かめられなければ例外）
    hu, ds = load_dicom_hu(dcm_path)
    # ② ウィンドウでクリップし、0〜1へ正規化（入門編のモダリティ別の前処理の章）
    lo, hi = window[0] - window[1]/2, window[0] + window[1]/2
    img = np.clip(hu, lo, hi)
    img = (img - lo) / (hi - lo)
    # ③ サイズをそろえ、テンソル化（チャネル次元を付ける）。ここでピクセル間隔の情報は失われるので、
    #    この画像は学習の入力用であって、大きさや距離を測る画像ではない（計測は元の格子と spacing で行う）
    img = cv2.resize(img.astype(np.float32), (size, size))
    return torch.from_numpy(img).unsqueeze(0)        # (1, H, W)

## 14.2　3次元：MONAIで読み込みから拡張までを一つにまとめる

In [ ]:
from monai import transforms as T

def build_pipeline(train=True):
    steps = [
        T.LoadImaged(keys=["image", "label"]),                       # NIfTI読み込み
        T.EnsureChannelFirstd(keys=["image", "label"]),
        T.Orientationd(keys=["image", "label"], axcodes="RAS"),      # ① 向きを統一
        T.Spacingd(keys=["image", "label"], pixdim=(1.5,)*3,
                   mode=("bilinear", "nearest")),                    # ② ボクセル間隔の統一
        T.ScaleIntensityRanged(keys="image", a_min=-100, a_max=240,
                               b_min=0.0, b_max=1.0, clip=True),     # ③ CT値のクリップと正規化
        T.CropForegroundd(keys=["image", "label"], source_key="image"),  # ④ 余白除去
    ]
    if train:                                                        # ⑤ 学習時のみ拡張
        steps += [
            T.SpatialPadd(keys=["image", "label"], spatial_size=(96,)*3, mode="constant"),
            T.RandCropByPosNegLabeld(keys=["image","label"], label_key="label",
                spatial_size=(96,)*3, pos=3, neg=1, num_samples=4),  # 前景中心を優先（前景に臓器も含まれる点に注意）
            # RAS前提なので spatial_axis=2 は頭尾方向の反転。実際には撮影されえない向きだが、
            # nnU-Net が既定で用いる鏡像反転にならった設定（第16章）。採否は検証データで確かめる。
            T.RandFlipd(keys=["image","label"], prob=0.5, spatial_axis=2),
            T.RandGaussianNoised(keys="image", prob=0.2),
        ]
    return T.Compose(steps)

train_tf, val_tf = build_pipeline(train=True), build_pipeline(train=False)

## 14.3　検証を、必ず挟む

In [ ]:
data = val_tf({"image": "case_001.nii.gz", "label": "case_001_seg.nii.gz"})  # 目視の確認は拡張なしのval_tfで（train_tfはパッチを複数返す）
img, lbl = data["image"][0], data["label"][0]
assert img.shape == lbl.shape, "画像とラベルの形が不一致"
print(f"値域: {img.min():.2f}〜{img.max():.2f}（0〜1のはず）")
# さらに、代表断面にマスクを重ねて目視（入門編の可視化の章の show_3d）